In [1]:
import duckdb

con = duckdb.connect()

path = "../../source_database/seed_data/stores.csv"

In [2]:
con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").show()

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ store_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_code    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ store_name    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ region        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ store_type    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ city          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ postcode      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ opened_date   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ acquired_from │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

In [3]:
# ── Completeness Audit: how many NULLs does each column have? ──

# Step 1: Get the list of column names automatically from the file.
# This way the same audit code works on ANY csv
columns_df = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
column_names = columns_df['column_name'].tolist()

# Step 2: Get the total row count once, so we can turn null counts into %.
total_rows = con.sql(f"SELECT COUNT(*) AS cnt FROM read_csv_auto('{path}')").df()['cnt'][0]
print(f"Total rows in file: {total_rows}")

# Step 3: For each column, ask DuckDB:
#   COUNT(*)        -> counts every row (NULL or not)
#   COUNT(col_name) -> counts only NON-null rows for that column
#   so COUNT(*) - COUNT(col_name) = how many rows are NULL in that column
null_report = []
for col in column_names:
    result = con.sql(f"""
        SELECT
            '{col}' AS column_name,
            COUNT(*) - COUNT({col}) AS null_count,
            ROUND(100.0 * (COUNT(*) - COUNT({col})) / COUNT(*), 2) AS null_percentage
        FROM read_csv_auto('{path}')
    """).df()
    null_report.append(result)

# Step 4: Stack all the per-column results into one table and show it.
import pandas as pd
null_report_df = pd.concat(null_report, ignore_index=True)
null_report_df

Total rows in file: 30


,column_name,null_count,null_percentage
0,store_id,0,0.0
1,store_code,0,0.0
2,store_name,0,0.0
3,region,0,0.0
4,store_type,0,0.0
5,city,0,0.0
6,postcode,0,0.0
7,country,0,0.0
8,opened_date,0,0.0
9,acquired_from,27,90.0


In [6]:
# Uniqueness Audit

# WHY: store_id is the primary key — must be unique or every join to this
# table (orders, inventory, etc.) silently fan-outs and double-counts.
# store_code is the human-facing business key — same rule applies.
# store_name + city isn't a hard key (a chain CAN have two branches in one
# city), but an EXACT duplicate of both is worth a human look — it might be
# the same store entered twice with two different store_ids.

dupe_id = con.sql(f"""
    SELECT store_id, COUNT(*) AS n FROM read_csv_auto('{path}')
    GROUP BY store_id HAVING COUNT(*) > 1
""").df()
print(f"Duplicate store_id: {len(dupe_id)}")
print(dupe_id)

dupe_code = con.sql(f"""
    SELECT store_code, COUNT(*) AS n FROM read_csv_auto('{path}')
    GROUP BY store_code HAVING COUNT(*) > 1
""").df()
print(f"\nDuplicate store_code: {len(dupe_code)}")
print(dupe_code)

dupe_name_city = con.sql(f"""
    SELECT store_name, city, COUNT(*) AS n, STRING_AGG(CAST(store_id AS VARCHAR), ', ') AS store_ids
    FROM read_csv_auto('{path}')
    GROUP BY store_name, city HAVING COUNT(*) > 1
""").df()
print(f"\nSame store_name + city appearing more than once (needs human judgement): {len(dupe_name_city)}")
print(dupe_name_city)

Duplicate store_id: 0
Empty DataFrame
Columns: [store_id, n]
Index: []

Duplicate store_code: 0
Empty DataFrame
Columns: [store_code, n]
Index: []

Same store_name + city appearing more than once (needs human judgement): 2
           store_name      city  n store_ids
0   NorthPeak Bristol   Bristol  2    13, 29
1  NorthPeak Brighton  Brighton  2     4, 28


In [8]:
# Referential integrity (reverse direction: does orders.csv point to real stores?)

# WHY: stores is the PARENT table here, so the direction flips — we're not
# checking stores against something else, we're checking whether every
# store_id referenced elsewhere (e.g. in orders) actually exists in stores.csv.
# Written defensively: only runs the join if orders.csv actually has a
# store_id column, instead of assuming the schema.

orders_path = "../../source_database/seed_data/orders.csv"
orders_cols = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{orders_path}')").df()['column_name'].tolist()

if 'store_id' in orders_cols:
    orphan_store_refs = con.sql(f"""
        SELECT COUNT(*) FROM read_csv_auto('{orders_path}') o
        LEFT JOIN read_csv_auto('{path}') s ON o.store_id = s.store_id
        WHERE s.store_id IS NULL
    """).fetchone()[0]
    print(f"orders.csv rows with a store_id that doesn't exist in stores.csv: {orphan_store_refs}")
else:
    print("orders.csv has no 'store_id' column — skip this check, or point it at whichever table references stores.")

orders.csv rows with a store_id that doesn't exist in stores.csv: 0


In [9]:
# store_code must follow the NP0XX pattern AND match store_id

# WHY: this is a structural rule specific to this table — NP001 should
# always correspond to store_id = 1, NP002 to store_id = 2, etc. If they
# ever drift apart, any report keying off store_code instead of store_id
# (common in business-facing exports) will join to the wrong store.

code_mismatch = con.sql(f"""
    SELECT store_id, store_code,
           TRY_CAST(regexp_extract(store_code, '(\\d+)', 1) AS INTEGER) AS code_number
    FROM read_csv_auto('{path}')
    WHERE store_code !~ '^NP\\d{{3}}$'
       OR TRY_CAST(regexp_extract(store_code, '(\\d+)', 1) AS INTEGER) != store_id
""").df()
print(f"store_code not matching 'NP0XX' pattern or not matching store_id: {len(code_mismatch)}")
print(code_mismatch)

store_code not matching 'NP0XX' pattern or not matching store_id: 0
Empty DataFrame
Columns: [store_id, store_code, code_number]
Index: []


In [10]:
# categorical columns: look for typos / inconsistent casing

# WHY: free-typed or loosely-validated categorical fields (region, store_type,
# country) are a common source of silent reporting errors — "Superstore" and
# "superstore" look the same to a human but count as two different groups in
# a GROUP BY. This doesn't assume what the "correct" list is — it just shows
# you every distinct value and its count, so you can eyeball it for typos.

for col in ['region', 'store_type', 'country', 'is_active', 'acquired_from']:
    print(f"\n=== distinct values: {col} ===")
    print(con.sql(f"""
        SELECT {col}, COUNT(*) AS n
        FROM read_csv_auto('{path}')
        GROUP BY {col}
        ORDER BY n DESC
    """).df())


=== distinct values: region ===
       region  n
0  South East  4
1   Yorkshire  4
2  South West  4
3    Scotland  4
4    Midlands  4
5        East  4
6       Wales  3
7  North West  3

=== distinct values: store_type ===
   store_type   n
0  superstore  19
1       local   8
2     express   3

=== distinct values: country ===
  country   n
0      UK  30

=== distinct values: is_active ===
   is_active   n
0       True  30

=== distinct values: acquired_from ===
  acquired_from   n
0           NaN  27
1     FreshMart   3


In [11]:
# postcode format (UK postcode pattern)

# WHY: a malformed postcode breaks geocoding/mapping and any delivery-radius
# or regional-reporting logic built on top of it.

bad_postcode = con.sql(f"""
    SELECT store_id, store_code, postcode
    FROM read_csv_auto('{path}')
    WHERE postcode !~ '^[A-Z]{{1,2}}[0-9][A-Z0-9]? [0-9][A-Z]{{2}}$'
""").df()
print(f"Postcodes not matching standard UK format: {len(bad_postcode)}")
print(bad_postcode)

Postcodes not matching standard UK format: 0
Empty DataFrame
Columns: [store_id, store_code, postcode]
Index: []


In [12]:
# opened_date sanity checks

# WHY: a store can't open in the future, and logically it can't have
# "opened" AFTER its record was created in the system (created_at) —
# that would mean the system knew about a store before it existed.

future_open = con.sql(f"""
    SELECT store_id, store_code, opened_date
    FROM read_csv_auto('{path}')
    WHERE opened_date > CURRENT_DATE
""").df()
print(f"opened_date in the future: {len(future_open)}")
print(future_open)

opened_after_created = con.sql(f"""
    SELECT store_id, store_code, opened_date, created_at
    FROM read_csv_auto('{path}')
    WHERE opened_date > CAST(created_at AS DATE)
""").df()
print(f"\nopened_date is AFTER created_at (impossible sequence): {len(opened_after_created)}")
print(opened_after_created)

opened_date in the future: 0
Empty DataFrame
Columns: [store_id, store_code, opened_date]
Index: []

opened_date is AFTER created_at (impossible sequence): 0
Empty DataFrame
Columns: [store_id, store_code, opened_date, created_at]
Index: []


In [13]:
# is_active must be a real boolean

# WHY: if is_active ever contains something other than True/False
# (e.g. "1"/"0", "Y"/"N", blank), any filter like WHERE is_active = TRUE
# will silently drop or keep rows it shouldn't.

bad_active = con.sql(f"""
    SELECT DISTINCT is_active FROM read_csv_auto('{path}')
    WHERE is_active NOT IN (TRUE, FALSE)
""").df()
print(f"is_active values that aren't a clean boolean: {len(bad_active)}")
print(bad_active)

is_active values that aren't a clean boolean: 0
Empty DataFrame
Columns: [is_active]
Index: []


In [14]:
# updated_at should never be before created_at

# WHY: basic logical ordering — a record can't be updated before it was
# created. Worth also checking whether EVERY updated_at equals created_at
# (seen in your sample) — that's not an error, but it tells you this table
# has never actually been updated since the initial load, which matters if
# you're about to build change-tracking/SCD logic on top of it.

bad_order = con.sql(f"""
    SELECT store_id, created_at, updated_at
    FROM read_csv_auto('{path}')
    WHERE updated_at < created_at
""").df()
print(f"updated_at earlier than created_at: {len(bad_order)}")
print(bad_order)

never_updated = con.sql(f"""
    SELECT COUNT(*) AS n_identical,
           (SELECT COUNT(*) FROM read_csv_auto('{path}')) AS total_rows
    FROM read_csv_auto('{path}')
    WHERE updated_at = created_at
""").df()
print("\nRows where updated_at == created_at (never updated since load):")
print(never_updated)

updated_at earlier than created_at: 0
Empty DataFrame
Columns: [store_id, created_at, updated_at]
Index: []

Rows where updated_at == created_at (never updated since load):
   n_identical  total_rows
0           30          30


In [15]:
# distribution: store counts by region and type

# WHY: not an error check — a sense check. A region or store_type with a
# suspiciously low/high count compared to the rest can mean a data load
# only partially completed, or a miscategorized store.

print("=== Stores per region ===")
print(con.sql(f"SELECT region, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY region ORDER BY n DESC").df())

print("\n=== Stores per store_type ===")
print(con.sql(f"SELECT store_type, COUNT(*) AS n FROM read_csv_auto('{path}') GROUP BY store_type ORDER BY n DESC").df())

print("\n=== Stores opened per year (growth trend) ===")
print(con.sql(f"""
    SELECT EXTRACT(YEAR FROM opened_date) AS year, COUNT(*) AS n_opened
    FROM read_csv_auto('{path}')
    GROUP BY year ORDER BY year
""").df())

=== Stores per region ===
       region  n
0    Midlands  4
1        East  4
2  South West  4
3   Yorkshire  4
4    Scotland  4
5  South East  4
6       Wales  3
7  North West  3

=== Stores per store_type ===
   store_type   n
0  superstore  19
1       local   8
2     express   3

=== Stores opened per year (growth trend) ===
    year  n_opened
0   2002         3
1   2003         4
2   2004         2
3   2005         3
4   2006         1
5   2007         1
6   2009         1
7   2010         3
8   2011         3
9   2014         2
10  2017         1
11  2018         2
12  2020         3
13  2022         1


In [16]:
# sentinel / placeholder values

# WHY: same reasoning as every other table — catches placeholder junk that
# passed the null check because it's not technically empty.

SENTINEL_VALUES = ['N/A', 'NA', 'NULL', 'null', '', '-', '?', 'unknown', 'UNKNOWN',
                    'TBD', '1900-01-01', '1970-01-01']

def sentinel_scan(con, path, sentinel_values):
    df = con.sql(f"SELECT * FROM read_csv_auto('{path}', ALL_VARCHAR=TRUE)").df()
    report = []
    for col in df.columns:
        counts = df[col].value_counts()
        hits = {v: int(counts[v]) for v in sentinel_values if v in counts.index}
        if hits:
            report.append({"column": col, "sentinel_hits": hits})
    return report

sentinel_report = sentinel_scan(con, path, SENTINEL_VALUES)
print("=== Sentinel value scan ===")
for r in sentinel_report:
    print(r)
if not sentinel_report:
    print("No sentinel values found from the standard list.")

=== Sentinel value scan ===
No sentinel values found from the standard list.


In [17]:
# Format consistency: Flag columns that contain mixed data types
def format_consistency_check(con, path):
    # Step 1: get every column name in the file
    schema = con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{path}')").df()
    columns = schema['column_name'].tolist()

    all_rows = []
    for col in columns:
        # Step 2: for THIS column, classify every value into a bucket
        breakdown = con.sql(f"""
            SELECT
                '{col}' AS column_name,
                CASE
                    WHEN "{col}" IS NULL OR TRIM("{col}") = '' THEN 'NULL/EMPTY'
                    WHEN TRY_CAST("{col}" AS BIGINT) IS NOT NULL THEN 'INTEGER'
                    WHEN TRY_CAST("{col}" AS DOUBLE) IS NOT NULL THEN 'DECIMAL'
                    WHEN TRY_CAST("{col}" AS DATE) IS NOT NULL THEN 'DATE'
                    WHEN TRY_CAST("{col}" AS TIMESTAMP) IS NOT NULL THEN 'TIMESTAMP'
                    WHEN lower("{col}") IN ('true','false') THEN 'BOOLEAN'
                    ELSE 'TEXT'
                END AS detected_type,
                COUNT(*) AS row_count
            FROM read_csv_auto('{path}', all_varchar=True)
            GROUP BY detected_type
            ORDER BY row_count DESC
        """).df()
        all_rows.append(breakdown)

    result = pd.concat(all_rows, ignore_index=True)

    # Step 3: flag a column only if it has 2+ real (non-null) types
    n_types = result[result['detected_type'] != 'NULL/EMPTY'].groupby('column_name')['detected_type'].transform('nunique')
    result['is_mixed'] = (n_types.reindex(result.index).fillna(0) > 1)
    return result

report = format_consistency_check(con, path)
report                       # full breakdown — bare, no print(), so it renders as a table

,column_name,detected_type,row_count,is_mixed
0,store_id,INTEGER,30,False
1,store_code,TEXT,30,False
2,store_name,TEXT,30,False
3,region,TEXT,30,False
4,store_type,TEXT,30,False
5,city,TEXT,30,False
6,postcode,TEXT,30,False
7,country,TEXT,30,False
8,opened_date,DATE,30,False
9,acquired_from,NULL/EMPTY,27,False
